# FULL ADJUSTABLE TRANSIT TOOL

### (fill in necessary links/values and run all)

In [ ]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
import sys
sys.path.append('../..')
from config import CENSUS_KEY
warnings.filterwarnings('ignore')
MODEL_PATH  = '../../models/xgb_model_2.json' 

CENSUS_VARIABLES = [
    'B01003_001E',  # total population
    'B08201_001E',  # total households
    'B08201_002E',  # zero-vehicle households
    'B08201_003E',  # one-vehicle households
    'B08301_001E',  # total commuters
    'B08301_002E',  # drove alone
    'B08301_004E',  # carpooled
    'B08301_010E',  # public transit
    'B23025_001E',  # population 16+ (labor force universe)
    'B23025_004E',  # employed civilians
    'B25003_001E',  # total occupied housing units
    'B25003_003E',  # renter-occupied
    'B19013_001E',  # median household income
]

COUNTIES = [
    #Add county FIPS codes here (last 3 digits of the 5-digit code)
]
GRID_SPACING_KM  = 0.5       # distance between candidate grid points
N_STATIONS       = 15        # station budget
MIN_SPACING_KM   = 3.0       # minimum distance between any two selected stations

MIN_JOB_KM = 4 #minimum distance between job centers

# BOUND BOX AROUND AREA OF INTEREST
BOUNDS = ()
#Example BOUNDS = (-123.55, 36.85, -121.20, 38.85) for Silicon Valley
#Or BOUNDS = (-122.75, 37.15, -121.55, 38.05) for San Francisco Bay Area
#Or BOUNDS = (-105.35, 39.55, -104.60, 40.25) for Denver Metro Area


STATE_CODE = '' # 2-digit state FIPS code (06 for California, 08 for Colorado etc)
STATE_ABBV = '' # LOWERCASE state abbreviation (ca for California, co for Colorado etc)
# Training features
FEATURE_COLS = [
    'Month',
    'Week',
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
    'is_post_covid',
    'dist_to_nearest_job_center_km'
]

#Census Features
CENSUS_FEATURE_COLS = [
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
]

### Pulling ACS Census Data

In [6]:
vars_str = ','.join(CENSUS_VARIABLES)
url = (
    f'https://api.census.gov/data/2023/acs/acs5'
    f'?get=NAME,{vars_str}'
    f'&for=tract:*'
    f'&in=state:{STATE_CODE}'
    f'&key={CENSUS_KEY}'
)

print('Pulling ACS 2023 data for state tracts...')
response = requests.get(url)
data = response.json()

state_tracts = pd.DataFrame(data[1:], columns=data[0])
state_tracts = state_tracts[state_tracts['county'].isin(COUNTIES)]

for col in CENSUS_VARIABLES:
    state_tracts[col] = pd.to_numeric(state_tracts[col], errors='coerce')
state_tracts.replace(-666666666, np.nan, inplace=True)

state_tracts['GEOID'] = state_tracts['state'] + state_tracts['county'] + state_tracts['tract']
print(f'Tracts loaded: {state_tracts.shape[0]}')
state_tracts.head()

Pulling ACS 2023 data for state tracts...
Tracts loaded: 408


,NAME,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E,state,county,tract,GEOID
7828,Census Tract 5001; Santa Clara County; California,7558,2962,178,1194,4706,3238,581,160,6272,4737,2962,1984,152188,06,085,500100,06085500100
7829,Census Tract 5002; Santa Clara County; California,5323,1937,128,1071,2747,1682,234,39,4709,2847,1937,1461,122238,06,085,500200,06085500200
7830,Census Tract 5003; Santa Clara County; California,4104,1791,61,842,2470,1416,145,209,3499,2547,1791,1115,137917,06,085,500300,06085500300
7831,Census Tract 5004; Santa Clara County; California,2651,1032,78,312,1529,1099,76,155,2200,1573,1032,604,151809,06,085,500400,06085500400
7832,Census Tract 5005; Santa Clara County; California,5661,2144,227,584,2910,1558,190,227,4808,2969,2144,443,177625,06,085,500500,06085500500


### Load TIGER tract boundaries and join demographics

In [7]:
print('Downloading TIGER 2023 tract boundaries (CA)...')
tract_url = 'https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_' + STATE_CODE + '_tract.zip'

state_tract_geo = gpd.read_file(tract_url)
state_tract_geo = state_tract_geo[state_tract_geo['COUNTYFP'].isin(COUNTIES)]
state_tract_geo = state_tract_geo.to_crs('EPSG:4326')
print(f'Tract geometries: {state_tract_geo.shape[0]}')

tracts = state_tract_geo[['GEOID', 'ALAND', 'geometry']].merge(
    state_tracts[['GEOID'] + CENSUS_VARIABLES],
    on='GEOID', how='left'
)
tracts.head()

Tract geometries: 408


,GEOID,ALAND,geometry,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E
0,06085503111,905170,"POLYGON ((-121.84812 37.32705, -121.84803 37.3...",5211,1141,7,201,2392,2198,432,16,4257,2556,1141,420,125521
1,06085503112,1136977,"POLYGON ((-121.88029 37.32271, -121.88012 37.3...",4069,1497,231,557,1880,1559,168,65,3249,2037,1497,1061,73419
2,06085506302,2014346,"POLYGON ((-121.9643 37.30184, -121.96429 37.30...",6352,2545,21,720,3784,2927,142,2,5351,3974,2545,1364,188705
3,06085511200,2059515,"POLYGON ((-122.1615 37.45371, -122.16114 37.45...",4877,1712,17,418,2380,995,27,23,3847,2403,1712,514,250001
4,06085506801,2892436,"POLYGON ((-121.97067 37.23454, -121.96884 37.2...",6233,2176,79,440,2855,1851,252,0,4790,2959,2176,716,244286


#### Engineer Model Features

In [8]:
def engineer_features(df):
    out = df.copy()
    out['population_density']    = out['B01003_001E'] / (out['ALAND'] / 1_000_000)
    out['low_car_rate']          = (out['B08201_002E'] + out['B08201_003E']) / out['B08201_001E']
    out['transit_commuter_rate'] = out['B08301_010E'] / out['B08301_001E']
    out['driver_rate']           = out['B08301_002E'] / out['B08301_001E']
    out['carpool_rate']          = out['B08301_004E'] / out['B08301_001E']
    out['employment_rate']       = out['B23025_004E'] / out['B23025_001E']
    out['renter_rate']           = out['B25003_003E'] / out['B25003_001E']
    out['median_income']         = out['B19013_001E']
    return out

tracts = engineer_features(tracts)
tracts[CENSUS_FEATURE_COLS].describe()

,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
count,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000
mean,3426.564948,0.338376,0.025541,0.710922,0.089535,0.640876,0.427621,163594.147059
std,2153.107254,0.164246,0.022377,0.125654,0.057054,0.077989,0.235733,54148.425871
min,0.756750,0.074899,0.000000,0.178358,0.000000,0.329582,0.038851,40167.000000
25%,2274.538438,0.206150,0.009247,0.630964,0.046617,0.595322,0.228395,119286.000000
50%,3202.801812,0.306889,0.021013,0.718643,0.078864,0.637222,0.396753,162321.000000
75%,4285.183641,0.456161,0.033702,0.807243,0.122205,0.686924,0.609078,203697.250000
max,20055.912453,0.927885,0.154550,0.960837,0.339578,0.887916,1.000000,250001.000000


In [9]:
def dist_to_point(lat, lon, clat, clon):
    R = 6371
    dlat, dlon = np.radians(lat - clat), np.radians(lon - clon)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat))*np.cos(np.radians(clat))*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def dist_to_nearest_job_center(lat, lon, centers):
    dists = [dist_to_point(lat, lon, c[0], c[1]) for c in centers]
    return min(dists)


### Load Job-center Data (LODES)

In [10]:
lodes_url = 'https://lehd.ces.census.gov/data/lodes/LODES8/' + STATE_ABBV + '/wac/' + STATE_ABBV + '_wac_S000_JT00_2021.csv.gz'

lodes = pd.read_csv(lodes_url, compression='gzip', dtype={'w_geocode': str})

# w_geocode is an 11-digit census block — first 11 chars = tract GEOID
lodes['GEOID'] = lodes['w_geocode'].str[:11]

# Aggregate jobs to tract level
jobs_by_tract = lodes.groupby('GEOID')['C000'].sum().reset_index()
jobs_by_tract.rename(columns={'C000': 'total_jobs'}, inplace=True)

print(jobs_by_tract.sort_values('total_jobs', ascending=False).head(10))

            GEOID  total_jobs
1977  06037207400      123890
6529  06073008511      106997
7061  06075011700       88928
2528  06037310400       85246
4600  06059075516       79766
4469  06059062610       78095
7980  06085505006       67760
7269  06075061501       61137
3848  06037980013       56240
2345  06037267901       54447


In [11]:
tracts = tracts.merge(jobs_by_tract, on='GEOID', how='left')
tracts['job_density'] = tracts['total_jobs'] / (tracts['ALAND'] / 1_000_000)

#### Filter Job Centers

In [12]:
emp_threshold = np.percentile(tracts['job_density'].dropna(), 95)
job_dense_tracts = tracts[tracts['job_density'] >= emp_threshold]
print(f'Tracts above 95th percentile of job density: {len(job_dense_tracts)}')

filtered_centers = []

for _, row in job_dense_tracts.sort_values('job_density', ascending=False).iterrows():
    lat = row.geometry.centroid.y
    lon = row.geometry.centroid.x
    
    if not filtered_centers:
        filtered_centers.append((lat, lon))
        continue
    
    # compute distance to all already accepted centers
    dists = [
        dist_to_point(lat, lon, c[0], c[1])
        for c in filtered_centers
    ]
    
    if min(dists) >= MIN_JOB_KM:
        filtered_centers.append((lat, lon))

job_centers = np.array(filtered_centers)

print(f'Filtered job centers: {len(job_centers)}')
print(job_centers[:10])


Tracts above 95th percentile of job density: 21
Filtered job centers: 8
[[  37.44800754 -122.16221258]
 [  37.34990879 -121.90216989]
 [  37.33036342 -122.00488747]
 [  37.4043783  -121.94659628]
 [  37.30610601 -121.92787199]
 [  37.3908184  -122.08045662]
 [  37.38005051 -122.00060213]
 [  37.36228654 -121.84664322]]


### Generate Candidate Grid

In [13]:
def generate_grid(bounds, spacing_km):
    min_lon, min_lat, max_lon, max_lat = bounds
    deg_lat = spacing_km / 110.574
    deg_lon = spacing_km / (111.320 * np.cos(np.radians((min_lat + max_lat) / 2)))
    lats = np.arange(min_lat, max_lat, deg_lat)
    lons = np.arange(min_lon, max_lon, deg_lon)
    grid_lons, grid_lats = np.meshgrid(lons, lats)
    pts = pd.DataFrame({'lon': grid_lons.ravel(), 'lat': grid_lats.ravel()})
    return gpd.GeoDataFrame(
        pts,
        geometry=[Point(x, y) for x, y in zip(pts.lon, pts.lat)],
        crs='EPSG:4326'
    )

grid = generate_grid(BOUNDS, GRID_SPACING_KM)
print(f'Total grid points: {len(grid)}')

# Spatial join: each grid point inherits features of the tract it falls inside
grid_joined = gpd.sjoin(
    grid,
    tracts[['GEOID', 'ALAND', 'geometry'] + CENSUS_FEATURE_COLS],
    how='left',
    predicate='within'
)
# Drop points outside county boundary
grid_joined = grid_joined.dropna(subset=['GEOID']).reset_index(drop=True)
grid_joined['is_post_covid'] = 1
grid_joined['Month']                   = 6
grid_joined['Week']                    = 24
print(f'Candidates inside Santa Clara County: {len(grid_joined)}')
grid_joined['dist_to_nearest_job_center_km'] = grid_joined.apply(
    lambda r: dist_to_nearest_job_center(r.lat, r.lon, job_centers), axis=1
)

# Fill sparse NaNs with tract medians
for col in FEATURE_COLS:
    grid_joined[col] = grid_joined[col].fillna(grid_joined[col].median())

grid_joined[['lon','lat'] + FEATURE_COLS].head()

Total grid points: 183402
Candidates inside Santa Clara County: 13335


,lon,lat,Month,Week,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income,is_post_covid,dist_to_nearest_job_center_km
0,-121.576177,36.895219,6,24,19.763696,0.14901,0.018593,0.678254,0.097413,0.625945,0.090803,245065.0,1,55.319884
1,-121.576177,36.899740,6,24,19.763696,0.14901,0.018593,0.678254,0.097413,0.625945,0.090803,245065.0,1,54.904821
2,-121.570489,36.899740,6,24,19.763696,0.14901,0.018593,0.678254,0.097413,0.625945,0.090803,245065.0,1,55.192944
3,-121.564800,36.899740,6,24,19.763696,0.14901,0.018593,0.678254,0.097413,0.625945,0.090803,245065.0,1,55.484157
4,-121.576177,36.904262,6,24,19.763696,0.14901,0.018593,0.678254,0.097413,0.625945,0.090803,245065.0,1,54.491236


## Predict Using Model

In [14]:
model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

X = grid_joined[FEATURE_COLS].values
grid_joined['predicted_ridership'] = np.expm1(model.predict(X))

print(f"Score range: {grid_joined['predicted_ridership'].min():.0f} – {grid_joined['predicted_ridership'].max():.0f}")
grid_joined[['lon','lat','predicted_ridership']].sort_values('predicted_ridership', ascending=False).head(10)

Score range: 4677 – 63916


,lon,lat,predicted_ridership
12398,-122.167755,37.442363,63916.207031
11894,-122.173443,37.424276,63916.207031
12273,-122.167755,37.437842,63916.207031
12146,-122.173443,37.433320,63916.207031
12145,-122.179132,37.433320,63916.207031
12020,-122.173443,37.428798,63916.207031
12019,-122.179132,37.428798,63916.207031
12018,-122.184820,37.428798,63916.207031
11893,-122.179132,37.424276,63916.207031
11767,-122.173443,37.419754,61367.882812


## Greedy Station Selection

In [15]:
def haversine(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2_arr)) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def greedy_select(candidates, n, min_spacing_km):
    ranked = candidates.sort_values('predicted_ridership', ascending=False)
    selected = []
    sel_lats, sel_lons = [], []

    for idx, row in ranked.iterrows():
        if len(selected) >= n:
            break
        if sel_lats:
            dists = haversine(row['lat'], row['lon'],
                              np.array(sel_lats), np.array(sel_lons))
            if dists.min() < min_spacing_km:
                continue
        selected.append(idx)
        sel_lats.append(row['lat'])
        sel_lons.append(row['lon'])

    return selected

selected_idx = greedy_select(grid_joined, N_STATIONS, MIN_SPACING_KM)
selected = grid_joined.loc[selected_idx].reset_index(drop=True)

print(f'Selected {len(selected)} stations')
print(f'Total predicted ridership: {selected["predicted_ridership"].sum():.0f}')
selected[['lat','lon','predicted_ridership'] + CENSUS_FEATURE_COLS].round(4)

Selected 15 stations
Total predicted ridership: 276741


,lat,lon,predicted_ridership,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
0,37.4424,-122.1678,63916.203125,1316.7822,0.9279,0.0343,0.1784,0.0448,0.4227,1.0000,44861.0
1,37.3338,-121.8777,23447.011719,20055.9125,0.6738,0.1546,0.4682,0.1347,0.4391,1.0000,40167.0
2,37.4062,-121.9402,16444.945312,9178.8148,0.7782,0.1038,0.4288,0.0651,0.8230,1.0000,209656.0
3,37.4062,-122.1109,15429.393555,6607.3766,0.6634,0.0486,0.5129,0.0445,0.7552,0.8032,150195.0
4,37.3655,-122.0312,15075.140625,4125.5732,0.6426,0.0876,0.5816,0.0929,0.7193,0.8005,166235.0
5,37.3203,-121.9118,15032.183594,3468.8792,0.6324,0.0894,0.6667,0.0946,0.7217,0.8402,131750.0
6,37.2751,-121.8378,14763.250977,11246.7494,0.6169,0.0322,0.7117,0.1577,0.7308,0.9828,120136.0
7,37.2977,-121.9516,14452.494141,12711.1773,0.4390,0.0000,0.8521,0.0144,0.6562,0.9402,87143.0
8,37.3384,-122.0369,14324.009766,12979.8813,0.3458,0.0197,0.8079,0.0786,0.7032,0.8247,118889.0
9,37.3655,-121.8492,14186.055664,6045.1390,0.5135,0.0292,0.8188,0.1951,0.4672,0.7915,65893.0


## Mapping

In [16]:
cmap = plt.cm.YlOrRd
vmin = selected['predicted_ridership'].min()
vmax = selected['predicted_ridership'].max()

def score_to_hex(val):
    norm = (val - vmin) / (vmax - vmin + 1e-9)
    r, g, b, _ = cmap(norm)
    return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

m = folium.Map(location=[37.33, -121.89], zoom_start=11, tiles='CartoDB positron')
# m = folium.Map(location=[39.85, -104.95], zoom_start=9, tiles='CartoDB positron')


# Background heatmap of all candidate scores
heat_data = grid_joined[['lat','lon','predicted_ridership']].values.tolist()
HeatMap(heat_data, radius=14, blur=18, min_opacity=0.3,
        gradient={'0.3':'blue','0.55':'lime','0.75':'orange','1.0':'red'}).add_to(m)

# Selected station markers
for i, row in selected.iterrows():
    color = score_to_hex(row['predicted_ridership'])
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=10,
        color='white', weight=2,
        fill=True, fill_color=color, fill_opacity=0.95,
        tooltip=(
            f"<b>Station {i+1}</b><br>"
            f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
            f"Pop density: {row['population_density']:.0f}/km²<br>"
            f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
            f"Low-car rate: {row['low_car_rate']:.1%}<br>"
            f"Employment rate: {row['employment_rate']:.1%}<br>"
            f"Renter rate: {row['renter_rate']:.1%}<br>"
            f"Median income: ${row['median_income']:,.0f}"
            f"Job proximity: {row['dist_to_nearest_job_center_km']:.1f} km"
        )
    ).add_to(m)

legend_html = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px'>
  <b>Predicted Ridership</b><br>
  <span style='color:#d73027'>●</span> High &nbsp;
  <span style='color:#fee08b'>●</span> Medium &nbsp;
  <span style='color:#4575b4'>●</span> Low<br><br>
  <i>Hover markers for tract features</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('transit_stations.html')
print('Saved → transit_stations.html')

Saved → transit_stations.html
